# EcoSort AI - Expanded Local Run

## 1. Check the active completed model

Read the active model pointer and its actual completion record. This notebook does not retrain. Historical results remain separate.

In [ ]:
from pathlib import Path
import json
PROJECT = Path.cwd()
assert (PROJECT / 'ecosort_runtime.py').exists(), 'Open this notebook from EcoSort-Local.'
active = json.loads((PROJECT / 'active_model.json').read_text())['results'] if (PROJECT / 'active_model.json').exists() else 'trained_expanded'
RESULTS = PROJECT / active
completion = json.loads((RESULTS / 'training_completed.json').read_text())
assert completion['status'] == 'completed'
completion

## 2. Load the selected model

Load the active 17-class classifier and validation-only calibration. The semantic model combines MobileNet with a frozen OpenCLIP encoder and a trained head. Letterboxing preserves the entire object. Print available devices; do not assume a GPU is present.

In [ ]:
import tensorflow as tf
from ecosort_runtime import EcoSort
print('GPU devices:', tf.config.list_physical_devices('GPU'))
engine = EcoSort()
print('Visual classes:', engine.classes)
print('Review threshold:', engine.settings['review_threshold'])

## 3. Open the English application

Reuse the running local application, or start it if absent. Classification remains separate from user-confirmed item guidance. Official City of Sydney and NSW EPA links accompany conditional disposal routes.

In [ ]:
from ecosort_runtime import application_running, launch
import webbrowser
if not application_running():
    demo = launch(prevent_thread_lock=True)
webbrowser.open('http://127.0.0.1:7861')

## 4. Read the final whole-photo evaluation

The thirty whole photos were frozen and duplicate-checked before their first prediction. The original ten photos and the 55 TACO diagnostic crops are regression/development sets, not new independent evidence. Report every case, including incorrect predictions and manual review. These convenience samples do not replace 50-100 local Sydney photos or guarantee performance in all scenes.

In [ ]:
import pandas as pd
report = json.loads((PROJECT / 'final_test/evaluation.json').read_text())
display(report)
display(pd.read_csv(PROJECT / 'final_test/predictions.csv')[['expected_label', 'predicted', 'confidence', 'manual_review', 'correct']])
display(json.loads((PROJECT / 'trained_semantic/taco_diagnostic/evaluation.json').read_text()))